# NFCorpus + Late Chunking

**Context-Enriched Retrieval for RAG — Iris.ai**

**Late Chunking** (Jina AI) embeds the **whole document first** to get contextualized token
embeddings, *then* applies chunk boundaries and pools tokens within each chunk's span. Each chunk
vector therefore carries surrounding-document context — and it needs **no LLM** (cheap).

## Model-fit caveat (important)
Late Chunking is designed for **bidirectional, mean-pooling** encoders. `Octen-Embedding-0.6B` is a
**causal (decoder) model with last-token pooling**, so:
- a chunk only sees *preceding* document context (causal attention), not the full document;
- the natural per-chunk vector is the **last token of the chunk's span** (matches the model's
  training), not a mean over the span.

So this is a **principled adaptation**, not textbook Late Chunking. We compute:
1. `baseline` — whole-doc embeddings (from the baseline notebook)
2. `naive-chunks` — each chunk embedded **independently** (no document context)
3. `late-chunks` — chunk vectors pooled from a **full-document forward pass** (contextualized)

`naive → late` isolates the context that late chunking adds on this model; `baseline → late` is the
total effect. Embedder `Octen-Embedding-0.6B` stays fixed; no generator is used.

## 1. Config & setup

In [1]:
import os, json, time
from pathlib import Path
import numpy as np
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
import pytrec_eval
from scipy import stats

MODEL_NAME   = 'Octen/Octen-Embedding-0.6B'
DATA_DIR     = Path('../nfcorpus')
SPLIT        = 'test'
BASELINE_DIR = Path('results/baseline')
OUT_DIR      = Path('results/late_chunking')
TOP_K        = 1000
BATCH_SIZE   = 32
SEED         = 42

# ---- knobs ---------------------------------------------------------------
CHUNK_TOKENS = 64            # chunk size in tokens (boundaries shared by both conditions)
POOL         = 'lasttoken'   # 'lasttoken' (matches model) or 'mean' (classic late chunking)
AGG          = 'max'         # pool chunk scores to a doc: 'max' or 'mean'
MAX_DOC_TOK  = 2048          # truncate very long docs for the full-doc forward pass

OUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, '| embedder:', MODEL_NAME, '| pooling:', POOL)

/Users/katerina_dimitrova/Downloads/Iris.ai RAG research/HyDE context Enrichment test/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps | embedder: Octen/Octen-Embedding-0.6B | pooling: lasttoken


## 2. Load corpus, queries, qrels

In [2]:
def load_jsonl(p):
    with open(p, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

def load_qrels(p):
    q = {}
    with open(p, encoding='utf-8') as f:
        next(f)
        for line in f:
            qid, did, s = line.rstrip('\n').split('\t')
            if int(s) > 0:
                q.setdefault(qid, {})[did] = int(s)
    return q

corpus = {r['_id']: ((r.get('title') or '') + '\n' + (r.get('text') or '')).strip()
          for r in load_jsonl(DATA_DIR / 'corpus.jsonl')}
corpus_ids = list(corpus.keys())
doc_index  = {d: i for i, d in enumerate(corpus_ids)}
qrels = load_qrels(DATA_DIR / 'qrels' / (SPLIT + '.tsv'))
all_q = {r['_id']: r['text'] for r in load_jsonl(DATA_DIR / 'queries.jsonl')}
queries = {q: all_q[q] for q in qrels if q in all_q}
query_ids = list(queries.keys())
print('corpus', len(corpus), 'docs | queries', len(queries))

corpus 3633 docs | queries 323


## 3. Compute naive-chunk and late-chunk embeddings

For each doc: tokenize once, split into `CHUNK_TOKENS` spans (shared boundaries). **Late** = pool the
full-doc forward pass per span; **naive** = decode each span to text and embed it independently.
Cached to `.npy` so you don't recompute.

In [3]:
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
model.max_seq_length = MAX_DOC_TOK
transformer = model[0]            # underlying Transformer module (returns token embeddings)
tok = model.tokenizer

LATE_PATH  = OUT_DIR / ('late_emb_%s_%d.npy' % (POOL, CHUNK_TOKENS))
NAIVE_PATH = OUT_DIR / ('naive_emb_%d.npy' % CHUNK_TOKENS)
OWNER_PATH = OUT_DIR / ('owner_%d.npy' % CHUNK_TOKENS)

if LATE_PATH.exists() and NAIVE_PATH.exists() and OWNER_PATH.exists():
    late_emb  = np.load(LATE_PATH)
    naive_emb = np.load(NAIVE_PATH)
    owner_idx = np.load(OWNER_PATH)
    print('loaded cached embeddings:', late_emb.shape)
else:
    late_vecs, naive_texts, owner = [], [], []
    for d in tqdm(corpus_ids):
        feats = model.preprocess([corpus[d]])
        feats = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in feats.items()}
        with torch.no_grad():
            out = transformer(feats)
        mask = out['attention_mask'][0].bool()
        tok_emb = out['token_embeddings'][0][mask].float()        # (T, H) real tokens only
        ids = feats['input_ids'][0][mask]
        T = tok_emb.shape[0]
        for start in range(0, T, CHUNK_TOKENS):
            end = min(start + CHUNK_TOKENS, T)
            v = tok_emb[start:end].mean(0) if POOL == 'mean' else tok_emb[end - 1]
            v = torch.nn.functional.normalize(v, dim=0)
            late_vecs.append(v.cpu().numpy())
            naive_texts.append(tok.decode(ids[start:end], skip_special_tokens=True))
            owner.append(doc_index[d])
    late_emb  = np.vstack(late_vecs).astype(np.float32)
    owner_idx = np.array(owner)
    print('embedding naive chunks independently...')
    naive_emb = model.encode(naive_texts, prompt_name='document', batch_size=BATCH_SIZE,
                             normalize_embeddings=True, convert_to_numpy=True,
                             show_progress_bar=True).astype(np.float32)
    np.save(LATE_PATH, late_emb); np.save(NAIVE_PATH, naive_emb); np.save(OWNER_PATH, owner_idx)
    print('done:', late_emb.shape, '| chunks/doc mean = %.2f' % (len(owner_idx) / len(corpus_ids)))

100%|██████████| 3633/3633 [10:32<00:00,  5.74it/s]


embedding naive chunks independently...


Batches: 100%|██████████| 688/688 [09:09<00:00,  1.25it/s] 


done: (21994, 1024) | chunks/doc mean = 6.05


## 4. Retrieve (pool chunks → docs) + evaluate

In [4]:
q_emb = model.encode([queries[q] for q in query_ids], prompt_name='query', batch_size=BATCH_SIZE,
                     normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
D = len(corpus_ids)
measures = {'ndcg_cut.1,3,5,10', 'recall.3,5,10,20,100', 'map', 'P.10', 'recip_rank', 'success.1,5,10'}
evaluator = pytrec_eval.RelevanceEvaluator(qrels, measures)

def pool_and_score(key_emb, owner):
    sims = q_emb @ key_emb.T
    Q = sims.shape[0]
    if AGG == 'max':
        scores = np.full((Q, D), -np.inf, dtype=np.float32)
        rows = np.broadcast_to(np.arange(Q)[:, None], sims.shape)
        cols = np.broadcast_to(owner[None, :], sims.shape)
        np.maximum.at(scores, (rows, cols), sims)
    else:
        scores = np.zeros((Q, D), dtype=np.float32); cnt = np.bincount(owner, minlength=D)
        for j, dd in enumerate(owner):
            scores[:, dd] += sims[:, j]
        scores /= np.maximum(cnt, 1)[None, :]
    k = min(TOP_K, D)
    topk = np.argpartition(-scores, k - 1, axis=1)[:, :k]
    run = {}
    for i, qid in enumerate(query_ids):
        idx = topk[i]; order = idx[np.argsort(-scores[i, idx])]
        run[qid] = {corpus_ids[j]: float(scores[i, j]) for j in order}
    pq = evaluator.evaluate(run)
    avg = lambda m: float(np.mean([pq[q][m] for q in pq]))
    metrics = {
        'NDCG@1': avg('ndcg_cut_1'), 'NDCG@3': avg('ndcg_cut_3'), 'NDCG@5': avg('ndcg_cut_5'), 'NDCG@10': avg('ndcg_cut_10'),
        'Recall@3': avg('recall_3'), 'Recall@5': avg('recall_5'), 'Recall@10': avg('recall_10'),
        'Recall@20': avg('recall_20'), 'Recall@100': avg('recall_100'),
        'Hit@1': avg('success_1'), 'Hit@5': avg('success_5'), 'Hit@10': avg('success_10'),
        'MAP': avg('map'), 'P@10': avg('P_10'), 'MRR': avg('recip_rank'),
    }
    return metrics, {q: pq[q]['ndcg_cut_10'] for q in pq}, run

naive_metrics, naive_pq, _       = pool_and_score(naive_emb, owner_idx)
late_metrics,  late_pq,  late_run = pool_and_score(late_emb,  owner_idx)
print('scored naive-chunks and late-chunks.')

scored naive-chunks and late-chunks.


## 5. Save + three-way comparison (baseline / naive-chunks / late-chunks)

In [5]:
with open(OUT_DIR / 'run.tsv', 'w') as f:
    for qid in query_ids:
        for rank, (did, sc) in enumerate(late_run[qid].items(), start=1):
            f.write('%s\tQ0\t%s\t%d\t%.6f\tlate_chunking\n' % (qid, did, rank, sc))
(OUT_DIR / 'metrics.json').write_text(json.dumps(late_metrics, indent=2))
(OUT_DIR / 'metrics_naive_chunks.json').write_text(json.dumps(naive_metrics, indent=2))
(OUT_DIR / 'per_query_ndcg10.json').write_text(json.dumps(late_pq, indent=2))
(OUT_DIR / 'config.json').write_text(json.dumps({
    'enrichment': 'Late Chunking (causal/last-token adaptation)', 'model_name': MODEL_NAME,
    'chunk_tokens': CHUNK_TOKENS, 'pool': POOL, 'agg': AGG, 'n_chunks': int(len(owner_idx)),
    'dataset': 'NFCorpus', 'split': SPLIT, 'n_queries': len(queries), 'seed': SEED,
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
}, indent=2))

base_metrics = json.loads((BASELINE_DIR / 'metrics.json').read_text())
base_pq = json.loads((BASELINE_DIR / 'per_query_ndcg10.json').read_text())

cols = [('baseline', base_metrics), ('naive-chunks', naive_metrics), ('late-chunks', late_metrics)]
print('%-12s' % 'metric' + ''.join('%14s' % n for n, _ in cols))
print('-' * 54)
for m in base_metrics:
    print('%-12s' % m + ''.join('%14.4f' % mt.get(m, float('nan')) for _, mt in cols))

def compare(name, pq, ref_pq, ref='baseline'):
    qs = [q for q in query_ids if q in ref_pq and q in pq]
    d = np.array([pq[q] - ref_pq[q] for q in qs])
    w, l = int((d > 1e-9).sum()), int((d < -1e-9).sum())
    nz = d[np.abs(d) > 1e-9]; p = stats.wilcoxon(nz).pvalue if len(nz) else 1.0
    sig = 'SIGNIFICANT' if p < 0.05 else 'ns'
    print('  %-12s vs %-12s dNDCG@10=%+.4f  wins/losses=%d/%d  Wilcoxon p=%.3g (%s)'
          % (name, ref, d.mean(), w, l, p, sig))

print('\nper-query NDCG@10 comparisons:')
compare('naive-chunks', naive_pq, base_pq)
compare('late-chunks',  late_pq,  base_pq)
compare('late-chunks',  late_pq,  naive_pq, ref='naive-chunks')   # isolates late-chunking effect

metric            baseline  naive-chunks   late-chunks
------------------------------------------------------
NDCG@1              0.4737        0.4752        0.4396
NDCG@3              0.4206        0.4159        0.4070
NDCG@5              0.4002        0.3910        0.3904
NDCG@10             0.3654        0.3561        0.3626
Recall@3            0.1112        0.1125        0.1114
Recall@5            0.1402        0.1391        0.1416
Recall@10           0.1747        0.1743        0.1819
Recall@20           0.2167        0.2160        0.2219
Recall@100          0.3424        0.3341        0.3419
Hit@1               0.4923        0.4892        0.4582
Hit@5               0.6842        0.6811        0.6780
Hit@10              0.7245        0.7214        0.7337
MAP                 0.1911        0.1865        0.1885
P@10                0.2687        0.2576        0.2697
MRR                 0.5851        0.5772        0.5654

per-query NDCG@10 comparisons:
  naive-chunks vs baseline     dN